# NLP Laboratory Assignment – 02
## Fundamental Language Processing for Three Different Languages

**Student:** Shravani Kurkute  
**Faculty:** Dr. Kapil N. Vhatkar

This notebook reproduces the exact pipeline implemented in `ASG2.py`. It imports the module functions so every number shown here is identical to the Excel workbook and the report. All values are computed at runtime.

### 0. Imports and NLTK data

In [ ]:
import os, sys
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import nltk
for pkg in ['punkt', 'stopwords']:
    try:
        nltk.data.find(pkg)
    except LookupError:
        nltk.download(pkg, quiet=True)
# Import the shared pipeline module so results match ASG2.py exactly.
import ASG2
print('pandas', pd.__version__)

### 1. Load and validate the dataset

In [ ]:
df = ASG2.load_dataset()
validation = ASG2.validate_dataset(df)
print('Columns:', validation['columns_found'])
print('Rows   :', validation['row_count'])
print('Missing:', validation['missing_values'])
print('Result :', 'PASSED' if validation['ok'] else 'FAILED')
df.head()

### 2. Stop-word resources
English uses NLTK's documented corpus. Hindi and Marathi use custom hand-curated lists (NLTK does not ship them); the full lists are shown in the `Stopword_Resources` worksheet and can be inspected below.

In [ ]:
print('English stop words from NLTK:', len(ASG2.get_stopwords('English')))
print('Hindi stop words (custom) :', len(ASG2.get_stopwords('Hindi')))
print('Marathi stop words (custom):', len(ASG2.get_stopwords('Marathi')))
print('Sample Hindi :', sorted(ASG2.get_stopwords('Hindi'))[:12])
print('Sample Marathi:', sorted(ASG2.get_stopwords('Marathi'))[:12])

### 3. Preprocessing functions
The same module functions are used for each language. English is lower-cased and NFKC-normalized; Hindi/Marathi use NFC and keep all Devanagari combining marks.

In [ ]:
examples = [
    ('English', 'I love natural language processing.'),
    ('Hindi', 'मुझे प्राकृतिक भाषा पसंद है।'),
    ('Marathi', 'मला नैसर्गिक भाषा आवडते.'),
]
for lang, text in examples:
    norm = ASG2.normalize_text(text, lang)
    dep = ASG2.remove_punctuation(norm, lang)
    toks = ASG2.tokenize_words(dep, lang)
    kept, removed = ASG2.remove_stopwords(toks, lang)
    stems = ASG2.apply_english_stemming(kept) if lang == 'English' else None
    print(lang)
    print('  normalized :', norm)
    print('  tokens     :', toks)
    print('  after stop :', kept)
    print('  removed    :', removed)
    print('  stems      :', stems)

### 4. Run the full pipeline for all three languages

In [ ]:
LANGUAGES = ['English', 'Hindi', 'Marathi']
results = {lang: ASG2.process_language(df, lang) for lang in LANGUAGES}
for lang in LANGUAGES:
    r = results[lang]
    print(lang, 'sentences', r['source_sentences'],
          'tokens_before', r['total_tokens_before'],
          'unique_before', r['unique_tokens_before'],
          'tokens_after', r['total_tokens_after'],
          'unique_after', r['unique_tokens_after'],
          'removed', r['tokens_removed'])

### 5. Comparative summary

In [ ]:
summary = ASG2.compare_languages(results, validation)
summary_df = pd.DataFrame(summary['languages']).T
summary_df

### 6. Frequency analysis (top 10 per language)

In [ ]:
for lang in LANGUAGES:
    print('\nTop 10 -', lang)
    for rank, (word, count) in enumerate(results[lang]['top10'], 1):
        print(f'  {rank:2}. {word} : {count}')

### 7. Charts
The same five charts are produced using the shared function. Hindi and Marathi labels are rendered with Pillow + Raqm complex shaping so the Devanagari text is correct.

In [ ]:
charts, dev_status = ASG2.generate_visualizations(results, summary)
print('Devanagari status:', dev_status)
for c in charts:
    print('  wrote', c)

### 8. Preview charts inline

In [ ]:
from IPython.display import Image, display
for name in ['token_comparison.png', 'vocabulary_comparison.png',
             'english_top10.png', 'hindi_top10.png', 'marathi_top10.png']:
    print(name)
    display(Image(filename=os.path.join('visualizations', name)))

### 9. Export results to Excel (nine sheets)

In [ ]:
xlsx = ASG2.export_results(results, summary, validation)
print('Excel written:', xlsx)
import openpyxl
print('Worksheets:', openpyxl.load_workbook(xlsx).sheetnames)

### 10. Self tests

In [ ]:
passed, failed, msgs = ASG2.run_self_tests()
for m in msgs:
    print(m)
print(f'\n{passed} passed, {failed} failed')

### 11. Conclusion
The notebook reproduces the full multilingual preprocessing pipeline with real computed statistics, charts and a nine-sheet Excel export. English uses Latin-script rules plus Porter stemming; Hindi and Marathi use NFC normalization, Devanagari-safe punctuation removal, Unicode-aware tokenization and language-specific stop words, without stemming.